# Day 17 · EDA 综合项目

> **前置**: Day11-16 已掌握 NumPy/Pandas/可视化/数据摄取
> **关键问题**: 面对一个陌生的数据集,如何从零开始“看懂”它? 本节用真实数据集走完 EDA(探索性数据分析)全流程 —— 这是数据科学家每天的工作。

**引入:像数据科学家一样思考**

抽问: pd.read_csv 的 na_values 参数有什么用? (指定哪些值视为缺失) response.json() 返回什么? (解析后的字典/列表) 前几节学了各种工具, 
今天把它们串起来 —— 从拿到数据到产出洞察报告。


**1. EDA 标准流程与数据加载**

EDA(Exploratory Data Analysis)标准 5 步流程: **看全貌** → **查缺失** → **找异常** → **修类型** → **造特征**。 
每一步都有对应的 Pandas/可视化方法。 本节用硬编码的 Titanic 风格数据集(25 条记录)完整演示。


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 硬编码 Titanic 风格数据集(含缺失值)
data = {
    "姓名": [
        "张三", "李四", "王五", "赵六", "钱七",
        "孙八", "周九", "吴十", "郑一", "王二",
        "刘三", "陈四", "杨五", "黄六", "林七",
        "何八", "高九", "罗十", "梁一", "宋二",
        "唐三", "韩四", "冯五", "董六", "萧七",
    ],
    "性别": [
        "男", "女", "男", "女", "男",
        "女", "男", "男", "女", "男",
        "女", "男", "女", "男", "女",
        "男", "女", "男", "女", "男",
        "女", "男", "女", "男", "女",
    ],
    "年龄": [
        22, 38, 26, 35, None,
        54, 2, 27, 14, 4,
        None, 31, 45, None, 19,
        28, 33, 42, None, 25,
        30, 24, 37, 48, 20,
    ],
    "舱位": [
        3, 1, 3, 1, 3,
        1, 3, 3, 2, 3,
        1, 3, 1, 3, 2,
        3, 2, 1, 3, 3,
        1, 3, 2, 1, 2,
    ],
    "票价": [
        7.25, 71.28, 7.92, 53.10, 8.05,
        51.86, 21.07, 11.13, 30.07, 16.70,
        26.55, 8.05, 89.10, 7.22, 13.00,
        7.75, 27.72, 42.40, 7.88, 9.50,
        77.28, 7.79, 61.98, 79.20, 12.50,
    ],
    "港口": [
        "S", "C", "S", "S", "S",
        "S", "S", "S", "C", "Q",
        "S", "S", "C", "S", None,
        "Q", "S", "C", "S", "S",
        "C", "S", None, "C", "S",
    ],
    "生存": [
        0, 1, 1, 1, 0,
        1, 0, 0, 1, 1,
        1, 0, 1, 0, 1,
        0, 1, 0, 0, 0,
        1, 0, 1, 1, 1,
    ],
}
df = pd.DataFrame(data)

# 1. 看全貌: shape 返回 (行数, 列数)
print("数据形状:", df.shape)
print("\n--- 前 5 行 ---")
print(df.head())
print("\n--- info(列类型与非空计数) ---")
df.info()
print("\n--- 缺失值统计 ---")
print(df.isnull().sum())


**2. 数据清洗:缺失值与异常值处理**

缺失值处理原则: **数值列用 median**(抗偏态), **分类列用 mode**(众数)。 pandas 2.x 的 Copy-on-Write 模式下, 
df[col].fillna(v, inplace=True) 不会改动原 DataFrame, 应该写成 df[col] = df[col].fillna(v)。 异常值用 
describe() 看四分位与极值, 用箱线图可视化识别。


In [ ]:
# 查看缺失值(清洗前)
print("清洗前缺失值:")
print(df.isnull().sum())

# 年龄(数值列)用中位数填充
# 注意 pandas 2.x 要赋值,不能写 inplace
med_age = df["年龄"].median()
print("年龄中位数:", med_age)
df["年龄"] = df["年龄"].fillna(med_age)

# 港口(分类列)用众数填充
mode_port = df["港口"].mode()[0]  # 取第一个
print("港口众数:", mode_port)
df["港口"] = df["港口"].fillna(mode_port)

# 验证: 缺失值应全部为 0
print("\n清洗后缺失值:")
print(df.isnull().sum())

# 异常值检测: describe 看四分位与极值
print("\n--- 数值列描述统计 ---")
print(df[["年龄", "票价"]].describe())

# 箱线图看票价异常值(高票价可能是离群点)
df[["年龄", "票价"]].boxplot()
plt.title("年龄与票价箱线图(异常值检测)")
# plt.show()


**3. 单变量分布可视化**

单变量分析回答“这一列长什么样”。 数值列用 **hist**(直方图看分布), 分类列用 **countplot**(计数图看各类占比)。 每张图配一句话结论。


In [ ]:
# 年龄分布直方图
df["年龄"].hist(bins=10, edgecolor="black")
plt.title("年龄分布直方图")
plt.xlabel("年龄")
plt.ylabel("人数")
# plt.show()

# 各舱位人数计数图
sns.countplot(x="舱位", data=df)
plt.title("各舱位人数")
# plt.show()

# 各性别人数计数图
sns.countplot(x="性别", data=df)
plt.title("各性别人数")
# plt.show()


**4. 多变量关系与生存率分析**

多变量分析回答“列与列之间有什么关系”。 核心问题: **舱位 / 性别是否影响生存率?** 用 groupby 聚合 + **barplot** 可视化 + 
**scatterplot** 看两数值变量关系。


In [ ]:
# 按舱位分组, 计算平均生存率
surv_by_class = df.groupby("舱位")["生存"].mean()
print("各舱位生存值:")
print(surv_by_class.round(2))

# 舱位生存率柱状图
sns.barplot(x="舱位", y="生存", data=df)
plt.title("各舱位生存率")
plt.ylabel("生存率")
# plt.show()

# 按性别分组, 计算平均生存率
surv_by_sex = df.groupby("性别")["生存"].mean()
print("\n各性别生存率:")
print(surv_by_sex.round(2))

# 性别生存率柱状图
sns.barplot(x="性别", y="生存", data=df)
plt.title("各性别生存率")
plt.ylabel("生存率")
# plt.show()

# 年龄 vs 票价散点图(按生存着色)
sns.scatterplot(
    x="年龄", y="票价", hue="生存", data=df
)
plt.title("年龄 vs 票价(按生存着色)")
# plt.show()


**5. 相关性热力图与洞察总结**

相关性热力图一次性看所有数值列之间的线性关系。 相关系数 r ∈ [-1, 1], |r| 越大相关性越强。 最后把发现整理成**量化洞察报告** —— 这是 EDA 的交付物。


In [ ]:
# 计算数值列相关系数矩阵
corr = df.corr(numeric_only=True)

# 热力图(annot 显示数值, cmap 配色)
sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("数值列相关性热力图")
# plt.show()

# 量化洞察报告(用数据说话)
c1 = df.groupby("舱位")["生存"].mean()
s1 = df.groupby("性别")["生存"].mean()
fare_s = df.groupby("生存")["票价"].mean()
print("=" * 40)
print("       EDA 洞察报告")
print("=" * 40)
print(
    f"头等舱生存率 {c1[1]:.0%}, "
    f"三等舱仅 {c1[3]:.0%}"
)
print(
    f"女性生存率 {s1['女']:.0%}, "
    f"男性仅 {s1['男']:.0%}"
)
print(
    f"生存者平均票价 {fare_s[1]:.1f}, "
    f"未生存者平均票价 {fare_s[0]:.1f}"
)
print("=" * 40)


**今日小结**

本节用 Titanic 数据集完整走完 EDA 5 步流程: **看全貌**(shape/info) → **查缺失**(isnull) → 
**找异常**(describe/boxplot) → **修类型/fillna** → **造特征/可视化**。 核心交付物是**量化洞察报告**。

**练习:**

- ⭐ 用 df.isnull().sum() 统计每列缺失值
- ⭐⭐ 画出“按港口分组生存率”柱状图并配结论
- ⭐⭐⭐ 新增“年龄段”特征(儿童/青年/中年/老年), 分析各年龄段生存率

**易错点:**

- pandas 2.x 中 df[col].fillna(v, inplace=True) 不生效, 要写 df[col] = df[col].fillna(v)
- 偏态数据用 **median** 不用 mean (避免极值干扰)
- 每张图配一句话结论, 图才有意义
